# Playground for trying to find data

In [1]:
from econdatapy import read
from fredapi import Fred
from dotenv import load_dotenv

import os
from pathlib import Path

import pandas as pd
import yfinance as yf

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

True

## EconData Data

Add `ECONDATA_CREDENTIALS=client_id;client_secret` to `.env` before running this cell.

In [2]:
sa_interest_rates = read.dataset(
    "MARKET_RATES",
    series_key="MMSD008"
)

sa_gdp = read.dataset(
   "NATL_ACC_SARB",
   series_key="KBP6006.R.S",
)

usd_zar_exch = read.dataset("MARKET_RATES", series_key="EXCX135")

sa_inflation = read.dataset("CPI_ANL_SERIES", series_key="CPS00000")

Fetching dataset(s) - MARKET_RATES

Processing data set: ECONDATA-MARKET_RATES-1.0.0

Fetching dataset(s) - NATL_ACC_SARB

Processing data set: ECONDATA-NATL_ACC_SARB-1.4.0

Fetching dataset(s) - MARKET_RATES

Processing data set: ECONDATA-MARKET_RATES-1.0.0

Fetching dataset(s) - CPI_ANL_SERIES

Processing data set: ECONDATA-CPI_ANL_SERIES-2.2.1



## FRED Data

In [3]:
os.getenv("FRED_API_KEY")

'a8a4a25d071ade00f10abf9d1f8b511a'

In [4]:
fred_api_key = os.getenv("FRED_API_KEY")
if not fred_api_key:
    raise RuntimeError("Add FRED_API_KEY to the project .env file")

fred = Fred(api_key=fred_api_key)

fred_series = {
    "us_fed_funds": "DFF",
    "us_5y_yield": "DGS5",
    "vix": "VIXCLS",
    "broad_usd_index": "DTWEXBGS",
    "iron_ore_usd_per_tonne": "PIORECRUSDM",
    "brent_usd_per_barrel": "DCOILBRENTEU",
}

fred_data = pd.concat(
    {name: fred.get_series(series_id) for name, series_id in fred_series.items()},
    axis=1,
).sort_index()
fred_data.index.name = "date"
fred_data.tail()

,us_fed_funds,us_5y_yield,vix,broad_usd_index,iron_ore_usd_per_tonne,brent_usd_per_barrel
date,,,,,,
2026-07-19,3.63,NaN,NaN,NaN,NaN,NaN
2026-07-20,3.63,4.33,18.65,NaN,NaN,86.99
2026-07-21,3.63,4.37,17.05,NaN,NaN,NaN
2026-07-22,3.63,4.41,16.64,NaN,NaN,NaN
2026-07-23,3.63,4.46,18.70,NaN,NaN,NaN


## Market Data

Yahoo Finance is used for the freely accessible gold and platinum continuous futures. `GC=F` and `PL=F` are futures proxies rather than physical spot fixings.

In [5]:
commodity_tickers = {
    "gold_usd_per_oz": "GC=F",
    "platinum_usd_per_oz": "PL=F",
}

commodity_data = yf.download(
    list(commodity_tickers.values()),
    start="2000-01-01",
    auto_adjust=False,
    progress=False,
)["Close"].rename(columns={ticker: name for name, ticker in commodity_tickers.items()})
commodity_data.index.name = "date"

commodity_returns = commodity_data.pct_change().add_suffix("_return")
commodity_data = commodity_data.join(commodity_returns)
commodity_data.tail()

Ticker,gold_usd_per_oz,platinum_usd_per_oz,gold_usd_per_oz_return,platinum_usd_per_oz_return
date,,,,
2026-07-20,4010.300049,1592.199951,-0.000598,-0.008469
2026-07-21,4071.100098,1626.099976,0.015161,0.021291
2026-07-22,4146.899902,1643.199951,0.018619,0.010516
2026-07-23,4046.600098,1599.099976,-0.024187,-0.026838
2026-07-24,4067.600098,1593.400024,0.005190,-0.003564


## Model-ready EconData Features

EconData returns a dataset dictionary. This helper selects its single requested series and converts the observation data to a date-indexed numeric series.

In [6]:
def econdata_series(dataset, name):
    observations = next(iter(dataset["data"].values())).copy()
    date_column = next(
        column for column in observations.columns
        if "time" in column.lower() or "date" in column.lower()
    )
    value_column = next(
        column for column in observations.columns
        if "value" in column.lower() or column.lower() in {"obs", "observation"}
    )
    series = pd.Series(
        pd.to_numeric(observations[value_column], errors="coerce").to_numpy(),
        index=pd.to_datetime(observations[date_column]),
        name=name,
    )
    return series[~series.index.duplicated(keep="last")].sort_index()


sa_repo_rate = econdata_series(sa_interest_rates, "sa_repo_rate")
sa_real_gdp = econdata_series(sa_gdp, "sa_real_gdp")
sa_cpi = econdata_series(sa_inflation, "sa_cpi")
sa_yoy_inflation = sa_cpi.pct_change(12).mul(100).rename("sa_yoy_inflation")

# Forward-fill sa_real_gdp only after reindexing onto the final daily model calendar.
# That makes each row use the latest observation known at that date.
usd_zar = econdata_series(usd_zar_exch, "usd_zar")
usd_zar_features = usd_zar.to_frame()
usd_zar_features["usd_zar_1w_return"] = usd_zar.pct_change(5)
usd_zar_features["usd_zar_1m_return"] = usd_zar.pct_change(21)
usd_zar_features["usd_zar_3m_return"] = usd_zar.pct_change(63)
usd_zar_features["usd_zar_1m_volatility"] = usd_zar.pct_change().rolling(21).std()
usd_zar_features.tail()

,usd_zar,usd_zar_1w_return,usd_zar_1m_return,usd_zar_3m_return,usd_zar_1m_volatility
TIME_PERIOD,,,,,
2026-07-13,16.3575,0.009050,-0.011165,-0.006366,0.005954
2026-07-14,16.4817,0.014977,0.013124,-0.002831,0.004947
2026-07-15,16.3712,0.000263,0.012443,0.001131,0.004994
2026-07-16,16.3425,-0.000599,0.006956,0.001195,0.004967
2026-07-17,16.4794,0.010355,0.006148,0.005958,0.004897


## Data to Download Manually

Download the historical tables as CSV files and save them under `data/manual/` with the filenames shown below.

- **SA 5-year yield** → [`sa_5y_yield.csv`](https://www.investing.com/rates-bonds/south-africa-5-year-bond-yield-historical-data). No reliable free API for the exact daily 5-year benchmark was identified. Set the date range, choose daily data, sign in to a free Investing.com account if prompted, then use **Download Data**. EconData's `CMJD002` is a 3–5 year average and is therefore not used as a substitute for the exact 5-year yield.
- **SA 5-year sovereign CDS** → [`sa_5y_cds.csv`](https://www.investing.com/rates-bonds/south-africa-cds-5-year-usd-historical-data). This is a free fallback when Bloomberg/Refinitiv is unavailable; keep the `Price` field as the CDS spread in bp.
- **Richards Bay coal (API 4)** → [`richards_bay_coal.csv`](https://za.investing.com/commodities/richards-bay-coal-futures-historical-data). Barchart may require an account/subscription for the historical download. The official benchmark is [Argus/McCloskey API 4](https://www.argusmedia.com/en/methodology/key-commodity-prices/argus-mccloskeys-api-4), but its history is licensed.

Investing.com CSVs are usually newest-first and may contain commas or percent signs; the loader below cleans those fields. Confirm the downloaded CDS `Price` units are basis points before modeling.

In [ ]:
MANUAL_DATA_DIR = PROJECT_ROOT / "data" / "manual"


def read_manual_market_csv(filename, value_name, value_column="Price"):
    path = MANUAL_DATA_DIR / filename
    if not path.exists():
        print(f"Not loaded: {path} (download it using the link in the cell above)")
        return pd.Series(name=value_name, dtype="float64")

    data = pd.read_csv(path)
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
    values = pd.to_numeric(
        data[value_column].astype(str).str.replace(",", "", regex=False).str.rstrip("%"),
        errors="coerce",
    )
    result = pd.Series(values.to_numpy(), index=data["Date"], name=value_name).dropna()
    return result[~result.index.duplicated(keep="last")].sort_index()


sa_5y_yield = read_manual_market_csv("sa_5y_yield.csv", "sa_5y_yield")
sa_5y_cds = read_manual_market_csv("sa_5y_cds.csv", "sa_5y_cds_bp")
richards_bay_coal = read_manual_market_csv(
    "richards_bay_coal.csv",
    "richards_bay_coal_usd",
    value_column="Last",
)

KeyError: 'Last'

## What Each DataFrame Contains

### Raw EconData responses

These four objects are EconData dataset dictionaries rather than ordinary DataFrames. Each contains metadata and a `data` dictionary holding the requested observation DataFrame. The `econdata_series` function extracts a clean, date-indexed Series from each object.

- `sa_interest_rates`: SARB series `MMSD008`, containing the South African repo rate as a percentage. The repo rate is the policy interest rate set by the South African Reserve Bank. A higher rate generally represents tighter monetary policy intended to restrain inflation; a lower rate generally supports borrowing and economic activity.
- `sa_gdp`: SARB series `6006.R.S`, containing quarterly, seasonally adjusted real GDP at 2015 prices in millions of rand. Real GDP measures the volume of goods and services produced after removing price changes. Growth indicates an expanding economy, while a decline indicates contraction. Because GDP is quarterly, the latest published value should be carried forward when it is placed on a daily model calendar.
- `sa_inflation`: Stats SA series `CPS00000`, containing the monthly headline Consumer Price Index. CPI measures the price level of a representative household consumption basket. The index level is converted into year-on-year inflation to make its rate of change easier to interpret.
- `usd_zar_exch`: SARB series `EXCX135`, containing the number of South African rand required to buy one US dollar. An increase means the rand has weakened against the dollar; a decrease means the rand has strengthened.

### Cleaned EconData series and derived FX features

- `sa_repo_rate`: cleaned daily repo-rate Series extracted from `sa_interest_rates`.
- `sa_real_gdp`: cleaned quarterly real-GDP Series extracted from `sa_gdp`. Changes in its level indicate real economic growth or contraction.
- `sa_cpi`: cleaned monthly headline-CPI index extracted from `sa_inflation`.
- `sa_yoy_inflation`: the percentage change in CPI relative to the same month one year earlier. A value of `5.0` means consumer prices are approximately 5% higher than a year ago.
- `usd_zar`: cleaned daily exchange-rate Series extracted from `usd_zar_exch`, quoted as rand per US dollar.
- `usd_zar_features`: contains the USD/ZAR level and its 5-, 21-, and 63-trading-day percentage returns, approximating one week, one month, and three months. Positive returns mean rand depreciation; negative returns mean rand appreciation. `usd_zar_1m_volatility` is the unannualised standard deviation of daily returns over the latest 21 observations. Higher volatility indicates larger or less predictable exchange-rate movements, regardless of direction.

### FRED data

`fred_data` combines series with different publication frequencies, so missing values between observations are expected and should not automatically be interpreted as missing source data.

- `us_fed_funds`: daily effective federal funds rate, expressed as a percentage. It represents the overnight rate at which US banks lend reserve balances to one another and is a key measure of US monetary-policy conditions.
- `us_5y_yield`: daily US 5-year constant-maturity Treasury yield, expressed as a percentage. It reflects expected US interest rates, inflation, and term compensation over roughly five years. Rising US yields can make dollar assets more attractive relative to emerging-market assets.
- `vix`: daily CBOE Volatility Index. It represents the options market's annualised expectation of S&P 500 volatility over the next 30 days. Higher values indicate greater expected uncertainty or risk aversion; it is not an asset return.
- `broad_usd_index`: trade-weighted broad US dollar index. A rising index means broad dollar appreciation, which often places pressure on emerging-market currencies such as the rand.
- `iron_ore_usd_per_tonne`: monthly global benchmark iron-ore price in US dollars per metric tonne. Iron ore is an important steelmaking input, so its price can proxy for global industrial and Chinese commodity demand.
- `brent_usd_per_barrel`: daily Brent crude-oil price in US dollars per barrel. Higher oil prices can worsen South Africa's import bill and increase inflation pressure because South Africa is a net oil importer.

### Other market and manually downloaded data

- `commodity_data`: daily closing prices for gold and platinum continuous futures, quoted in US dollars per troy ounce, together with their one-session decimal returns. A return of `0.01` means a 1% price increase. These are futures proxies rather than physical spot fixings. Gold often behaves as a safe-haven asset, while platinum is more exposed to industrial and automotive demand; both are important South African mineral exports.
- `sa_5y_yield`: daily South African 5-year government-bond yield, expressed as a percentage. It represents the government's approximate annual borrowing cost at the five-year maturity. A higher yield can reflect expectations of higher repo rates, inflation, or South African risk. Comparing it with the equivalent US yield helps measure relative interest-rate support for the rand.
- `sa_5y_cds`: daily South African 5-year sovereign credit-default-swap spread in basis points. It is the approximate annual insurance premium for protection against a South African sovereign default. One basis point is 0.01 percentage points, so 200 bp is approximately 2% per year. A wider spread indicates greater perceived sovereign credit risk.
- `richards_bay_coal`: Richards Bay API 4 thermal-coal benchmark price, normally quoted in US dollars per metric tonne. Higher prices can improve South African export revenues and the trade balance, although the effect also depends on export volumes and transport capacity.

## Save Retrieved Data

The cell below creates `data/raw/` and saves every cleaned dataset as a CSV. Manual series that have not been downloaded yet are skipped instead of creating empty files.

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

sa_econdata = pd.concat(
    [sa_repo_rate, sa_real_gdp, sa_cpi, sa_yoy_inflation, usd_zar_features],
    axis=1,
).sort_index()

datasets_to_save = {
    "sa_econdata.csv": sa_econdata,
    "fred_data.csv": fred_data,
    "commodity_data.csv": commodity_data,
    "sa_5y_yield.csv": sa_5y_yield,
    "sa_5y_cds.csv": sa_5y_cds,
    "richards_bay_coal.csv": richards_bay_coal,
}

saved_files = []
for filename, data in datasets_to_save.items():
    if data.empty:
        print(f"Skipped {filename}: no data loaded")
        continue
    output_path = RAW_DATA_DIR / filename
    data.to_csv(output_path, index=True, index_label="date")
    saved_files.append(output_path)

saved_files